# Batch AIS extraction with Docling



Convert every PDF in `../AIS-documents` with Docling and create MongoDB-ready clause documents. Numbered headings are derived independently from each standard instead of using an AIS-002-specific title map.



Every output record includes `AIS_id`, the canonical AIS document identifier for the source rule. `source_file` keeps revisions and parts distinct:



```json

{

  "AIS_id": "AIS-032",

  "source_file": "AIS-032.pdf",

  "heading": "EXTENSION OF APPROVAL",

  "heading_number": "10",

  "subheading": "Changes in Technical Specifications",

  "rule": "10.1.1",

  "description": "Every modification ...",

  "AIS": ["AIS-001"]

}

```



Outputs are written to `ais_extracted/`: one JSON file per PDF, `all_ais_mongodb_documents.json`, and `extraction_manifest.json`. Set `MAX_FILES` for a smaller run. MongoDB writes remain disabled until `WRITE_TO_MONGODB = True`.

In [60]:
from __future__ import annotations



import json

import os

import re

import unicodedata

from collections import defaultdict

from pathlib import Path

from typing import Any



from dotenv import load_dotenv

from pydantic import BaseModel, ConfigDict, Field

from pymongo import ASCENDING, MongoClient, UpdateOne



from ingest_regulatory_graph import (

    DocumentKind,

    build_converter,

    convert_document,

    iter_regulatory_chunks,

)



load_dotenv()



AIS_DOCS_DIR = Path("../AIS-documents").resolve()

OUTPUT_DIR = Path("ais_extracted").resolve()

COMBINED_OUTPUT_PATH = OUTPUT_DIR / "all_ais_mongodb_documents.json"

MANIFEST_PATH = OUTPUT_DIR / "extraction_manifest.json"

ENABLE_OCR = True

MAX_FILES: int | None = None  # Set to a small integer for a smoke run.



# MongoDB writes are deliberately opt-in.

WRITE_TO_MONGODB = True

MONGODB_URI = os.getenv("MONGODB_URI")

MONGODB_DATABASE = os.getenv("MONGODB_DATABASE", "automotive_regulations")

MONGODB_COLLECTION = os.getenv("MONGODB_COLLECTION", "ais_rules")



AIS_ID_RE = re.compile(r"AIS[\s_-]*(\d{1,3})(?!\d)", re.IGNORECASE)





def source_ais_from_path(pdf_path: Path) -> str:

    match = AIS_ID_RE.search(pdf_path.stem)

    if not match:

        raise ValueError(f"Could not identify an AIS number from {pdf_path.name}")

    return f"AIS-{match.group(1).zfill(3)}"





assert AIS_DOCS_DIR.is_dir(), f"AIS documents folder not found: {AIS_DOCS_DIR}"

pdf_paths = sorted(

    (path for path in AIS_DOCS_DIR.iterdir() if path.is_file() and path.suffix.lower() == ".pdf"),

    key=lambda path: path.name.casefold(),

)

if MAX_FILES is not None:

    pdf_paths = pdf_paths[:MAX_FILES]

assert pdf_paths, f"No PDF files found in {AIS_DOCS_DIR}"

unidentified_paths = [path.name for path in pdf_paths if not AIS_ID_RE.search(path.stem)]

assert not unidentified_paths, f"AIS number missing from filenames: {unidentified_paths}"

print(f"Discovered {len(pdf_paths)} AIS PDFs in {AIS_DOCS_DIR}")

Discovered 43 AIS PDFs in /Users/utsav.talwar/Desktop/UST/AIS-documents


In [61]:
class AISRuleDocument(BaseModel):

    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)



    AIS_id: str = Field(pattern=r"^AIS-\d{3}$")

    source_file: str = Field(min_length=1)

    heading: str = Field(min_length=1)

    heading_number: str = Field(min_length=1)

    subheading: str | None = None

    rule: str = Field(pattern=r"^(?:[A-Z]-)?\d+(?:\.\d+)*$")

    description: str = Field(min_length=1)

    AIS: list[str] | None = None





NUMBERED_HEADING_RE = re.compile(

    r"^[\s\-–—•*]*(?P<number>(?:[A-Z]\s*[-.]\s*)?\d+(?:\.\d+)*)"

    r"\.?\s+(?P<title>[A-Za-z][^\n]{0,240})$",

    re.IGNORECASE,

)

CLAUSE_PATH_RE = re.compile(

    r"^Clause\s+((?:[A-Z]-)?\d+(?:\.\d+)*)$", re.IGNORECASE

)

LEADING_CLAUSE_RE = re.compile(

    r"^[\s\-–—•*]*(?:[A-Z]\s*[-.]\s*)?\d+(?:\.\d+)*[.,]?\s*",

    re.IGNORECASE,

)

EMBEDDED_CLAUSE_RE = re.compile(

    r"(?:^|\s)[\-–—•*]\s*((?:[A-Z]\s*[-.]\s*)?\d+(?:\.\d+)+)\s+",

    re.IGNORECASE,

)

AIS_REFERENCE_RE = re.compile(

    r"\bAIS\s*(?:No\.?\s*)?[-:/]?\s*(\d{3})\b", re.IGNORECASE

)





def normalize_text(value: str) -> str:

    value = unicodedata.normalize("NFKC", value)

    value = value.replace("\u00ad", "").replace("\u200b", "")

    return re.sub(r"\s+", " ", value).strip()





def canonical_clause(value: str) -> str:

    compact = re.sub(r"\s+", "", value.upper()).rstrip(".")

    if re.match(r"^[A-Z]\.", compact):

        compact = f"{compact[0]}-{compact[2:]}"

    return compact





def parse_numbered_heading(value: str) -> tuple[str, str] | None:

    match = NUMBERED_HEADING_RE.match(normalize_text(value))

    if not match:

        return None

    number = canonical_clause(match.group("number"))

    title = match.group("title").strip(" .:-")

    if not title or len(title) > 240:

        return None

    return number, title





def collect_heading_titles(docling_document: Any) -> dict[str, str]:

    titles: dict[str, str] = {}

    for chunk in iter_regulatory_chunks(

        docling_document,

        document_kind=DocumentKind.STANDARD,

        max_chunk_chars=50_000,

        chunk_overlap=0,

    ):

        candidates = list(chunk.docling_headings)

        if len(normalize_text(chunk.text)) <= 180:

            candidates.append(chunk.text)

        for candidate in candidates:

            parsed = parse_numbered_heading(candidate)

            if parsed:

                titles.setdefault(*parsed)

    return titles





def extract_ais_references(text: str, source_ais: str) -> list[str] | None:

    references = {

        f"AIS-{match.group(1)}" for match in AIS_REFERENCE_RE.finditer(text)

    }

    references.discard(source_ais)

    return sorted(references) or None





def nearest_subheading(rule: str, heading_titles: dict[str, str]) -> str | None:

    parts = rule.split(".")

    for index in range(len(parts), 1, -1):

        candidate = ".".join(parts[:index])

        if candidate in heading_titles:

            return heading_titles[candidate]

    return None





def apply_source_amendments(

    source_ais: str, source_file: str, rule: str, description: str

) -> str:

    is_legacy_ais_002 = (

        source_ais == "AIS-002"

        and source_file.casefold() == "5192015105902amais_002.pdf".casefold()

        and rule == "5.1"

    )

    if not is_legacy_ais_002:

        return description



    description = re.sub(

        r"N\s*1\s+with\s+GVW\s+not\s+exceeding\s+2t\s*&\s*M\s*1",

        "N1 with GVW not exceeding 2 T, M1 and L7",

        description,

        flags=re.IGNORECASE,

    )

    return (

        f"{description} Amendment No. 1 dated 18 May 2015 substitutes the "

        "clause 5.1 row with N1 having GVW not exceeding 2 T, M1 and L7; "

        "mandatory field of vision (F2 + F1) or (F2 + F3); additional "

        "options F3 or F1."

    )

In [62]:
def extract_clause_parts(

    docling_document: Any,

    heading_titles: dict[str, str],

) -> dict[str, list[str]]:

    clause_parts: dict[str, list[str]] = defaultdict(list)

    known_top_levels = {number.split(".", maxsplit=1)[0] for number in heading_titles}



    def append_part(rule: str | None, text: str) -> None:

        if not rule:

            return

        rule = canonical_clause(rule)

        top_level = rule.split(".", maxsplit=1)[0]

        if known_top_levels and top_level not in known_top_levels:

            return

        cleaned = normalize_text(text)

        if cleaned and (not clause_parts[rule] or clause_parts[rule][-1] != cleaned):

            clause_parts[rule].append(cleaned)



    for chunk in iter_regulatory_chunks(

        docling_document,

        document_kind=DocumentKind.STANDARD,

        max_chunk_chars=50_000,

        chunk_overlap=0,

    ):

        # Annex numbering needs a separate schema; do not mix it with body rules.

        if any(level.upper().startswith("ANNEX") for level in chunk.hierarchy):

            continue



        rule_match = CLAUSE_PATH_RE.fullmatch(chunk.rule or "")

        rule = canonical_clause(rule_match.group(1)) if rule_match else None

        text = normalize_text(chunk.text)



        table_rule_match = re.match(

            r"^[\s\-–—•*]*((?:[A-Z]\s*[-.]\s*)?\d+(?:\.\d+)+),",

            text,

            re.IGNORECASE,

        )

        if table_rule_match:

            rule = canonical_clause(table_rule_match.group(1))



        explicit_number = NUMBERED_HEADING_RE.match(text)

        if not explicit_number and not table_rule_match:

            for heading in reversed(chunk.docling_headings):

                parsed = parse_numbered_heading(heading)

                if parsed:

                    heading_rule = parsed[0]

                    if not rule or not rule.startswith(f"{heading_rule}."):

                        rule = heading_rule

                    break



        boundaries = list(EMBEDDED_CLAUSE_RE.finditer(text))

        if not boundaries:

            append_part(rule, text)

            continue



        if boundaries[0].start() > 0:

            append_part(rule, text[:boundaries[0].start()])

        for index, boundary in enumerate(boundaries):

            end = boundaries[index + 1].start() if index + 1 < len(boundaries) else len(text)

            append_part(boundary.group(1), text[boundary.start():end])



    return dict(clause_parts)





def clause_sort_key(rule: str) -> tuple[tuple[int, int | str], ...]:

    return tuple(

        (0, int(token)) if token.isdigit() else (1, token)

        for token in re.findall(r"[A-Z]+|\d+", rule.upper())

    )





def build_mongodb_documents(

    clause_parts: dict[str, list[str]],

    heading_titles: dict[str, str],

    source_ais: str,

    source_file: str,

) -> list[dict[str, Any]]:

    documents: list[AISRuleDocument] = []



    for rule, parts in clause_parts.items():

        heading_number = rule.split(".", maxsplit=1)[0]

        description = LEADING_CLAUSE_RE.sub(

            "", normalize_text(" ".join(parts)), count=1

        )

        description = apply_source_amendments(

            source_ais, source_file, rule, description

        )

        if not description:

            continue



        documents.append(

            AISRuleDocument(

                AIS_id=source_ais,

                source_file=source_file,

                heading=heading_titles.get(

                    heading_number, f"SECTION {heading_number}"

                ),

                heading_number=heading_number,

                subheading=nearest_subheading(rule, heading_titles),

                rule=rule,

                description=description,

                AIS=extract_ais_references(description, source_ais),

            )

        )



    documents.sort(key=lambda item: clause_sort_key(item.rule))

    return [document.model_dump() for document in documents]





def extract_pdf(converter: Any, pdf_path: Path) -> list[dict[str, Any]]:

    source_ais = source_ais_from_path(pdf_path)

    docling_document = convert_document(converter, pdf_path, page_range=None)

    heading_titles = collect_heading_titles(docling_document)

    clause_parts = extract_clause_parts(docling_document, heading_titles)

    return build_mongodb_documents(

        clause_parts, heading_titles, source_ais, pdf_path.name

    )

In [63]:
heading_titles = {

    "3": "TYPE APPROVAL",

    "3.2": "Modifications in Technical Specifications",

}

sample_parts = {

    "3.2.1": [

        "3.2.1 Every modification shall be intimated to the test agency and "

        "checked against AIS No 001 and AIS-037."

    ],

    "3.2.1.1": ["3.2.1.1 The modified component still complies."],

}

sample_documents = build_mongodb_documents(

    sample_parts,

    heading_titles,

    source_ais="AIS-037",

    source_file="AIS-037.pdf",

)



assert [document["rule"] for document in sample_documents] == ["3.2.1", "3.2.1.1"]

assert sample_documents[0]["AIS_id"] == "AIS-037"

assert "source_ais" not in sample_documents[0]

assert sample_documents[0]["source_file"] == "AIS-037.pdf"

assert sample_documents[0]["heading"] == "TYPE APPROVAL"

assert sample_documents[0]["subheading"] == "Modifications in Technical Specifications"

assert sample_documents[0]["AIS"] == ["AIS-001"]

assert sample_documents[1]["description"].startswith("The modified component")

print("Focused AIS_id schema check passed.")

Focused AIS_id schema check passed.


In [64]:
def output_path_for_pdf(pdf_path: Path) -> Path:

    safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", pdf_path.stem).strip("_")

    return OUTPUT_DIR / f"{safe_stem}.json"





def run_batch(

    paths: list[Path],

    *,

    write_files: bool = True,

) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:

    converter = build_converter(enable_ocr=ENABLE_OCR)

    ocr_converter = converter if ENABLE_OCR else None

    all_documents: list[dict[str, Any]] = []

    manifest: list[dict[str, Any]] = []

    if write_files:

        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



    for index, pdf_path in enumerate(paths, start=1):

        print(f"[{index}/{len(paths)}] {pdf_path.name}")

        try:

            documents = extract_pdf(converter, pdf_path)

            used_ocr = ENABLE_OCR

            if not documents and not ENABLE_OCR:

                print("  No clauses found; retrying with OCR.")

                if ocr_converter is None:

                    ocr_converter = build_converter(enable_ocr=True)

                documents = extract_pdf(ocr_converter, pdf_path)

                used_ocr = True



            all_documents.extend(documents)

            entry = {

                "AIS_id": source_ais_from_path(pdf_path),

                "source_file": pdf_path.name,

                "status": "ok" if documents else "no_clauses",

                "document_count": len(documents),

                "used_ocr": used_ocr,

                "error": None,

            }

            if write_files:

                output_path_for_pdf(pdf_path).write_text(

                    json.dumps(documents, indent=2, ensure_ascii=False),

                    encoding="utf-8",

                )

        except Exception as error:

            entry = {

                "AIS_id": None,

                "source_file": pdf_path.name,

                "status": "error",

                "document_count": 0,

                "used_ocr": False,

                "error": f"{type(error).__name__}: {error}",

            }

            print(f"  ERROR: {entry['error']}")

        manifest.append(entry)



    return all_documents, manifest

In [65]:
# Convert two differently numbered standards and one image-only PDF.

smoke_paths = [

    next(path for path in pdf_paths if source_ais_from_path(path) == "AIS-022"),

    next(path for path in pdf_paths if source_ais_from_path(path) == "AIS-055"),

]

smoke_documents, smoke_manifest = run_batch(smoke_paths, write_files=False)



assert all(entry["status"] != "error" for entry in smoke_manifest), smoke_manifest

assert all(entry["document_count"] > 0 for entry in smoke_manifest), smoke_manifest

assert {document["AIS_id"] for document in smoke_documents} == {"AIS-022", "AIS-055"}

assert all("source_ais" not in document for document in smoke_documents)

assert len({(document["source_file"], document["rule"]) for document in smoke_documents}) == len(smoke_documents)



scanned_path = next(path for path in pdf_paths if source_ais_from_path(path) == "AIS-015")

scanned_documents, scanned_manifest = run_batch([scanned_path], write_files=False)

assert scanned_manifest[0]["status"] != "error", scanned_manifest

assert scanned_manifest[0]["used_ocr"] is True

print(

    f"Smoke checks passed: {len(smoke_documents)} text-PDF records and "

    f"{len(scanned_documents)} OCR records."

)

[INFO] 2026-07-29 14:03:26,202 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 14:03:26,203 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 14:03:26,213 [RapidOCR] download_file.py:60: File exists and is valid: /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-29 14:03:26,214 [RapidOCR] main.py:50: Using /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-29 14:03:26,315 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 14:03:26,315 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 14:03:26,317 [RapidOCR] download_file.py:60: File exists and is valid: /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-29 14:03:26,317 [RapidOCR] main.py:50: Using /Users/utsav.talwa

[1/2] 10172013104127AMAIS-022andAmds.pdf


[INFO] 2026-07-29 14:03:26,593 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 14:03:26,594 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 14:03:26,615 [RapidOCR] download_file.py:60: File exists and is valid: /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth
[INFO] 2026-07-29 14:03:26,616 [RapidOCR] main.py:50: Using /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1607.69it/s]


[2/2] 1122018124450PMAIS_55_and_Amd_1.pdf


[INFO] 2026-07-29 14:03:52,943 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 14:03:52,944 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 14:03:52,952 [RapidOCR] download_file.py:60: File exists and is valid: /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-29 14:03:52,953 [RapidOCR] main.py:50: Using /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-29 14:03:53,062 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 14:03:53,062 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 14:03:53,065 [RapidOCR] download_file.py:60: File exists and is valid: /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-29 14:03:53,065 [RapidOCR] main.py:50: Using /Users/utsav.talwa

[1/1] 5~15~2008~11~39~25~AM16 AIS-015.PDF


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 3083.21it/s]


Smoke checks passed: 36 text-PDF records and 0 OCR records.


In [66]:
all_documents, extraction_manifest = run_batch(pdf_paths)



record_keys = [

    (document["source_file"], document["rule"])

    for document in all_documents

]

assert len(record_keys) == len(set(record_keys)), "Duplicate source_file/rule records found"



COMBINED_OUTPUT_PATH.write_text(

    json.dumps(all_documents, indent=2, ensure_ascii=False),

    encoding="utf-8",

)

MANIFEST_PATH.write_text(

    json.dumps(extraction_manifest, indent=2, ensure_ascii=False),

    encoding="utf-8",

)



status_counts = {

    status: sum(entry["status"] == status for entry in extraction_manifest)

    for status in ("ok", "no_clauses", "error")

}

print(

    f"Wrote {len(all_documents)} documents from {len(pdf_paths)} PDFs to "

    f"{OUTPUT_DIR.name}/. Status: {status_counts}"

)

display(extraction_manifest)

[INFO] 2026-07-29 14:04:00,364 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 14:04:00,365 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 14:04:00,371 [RapidOCR] download_file.py:60: File exists and is valid: /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-29 14:04:00,372 [RapidOCR] main.py:50: Using /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-29 14:04:00,467 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 14:04:00,468 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 14:04:00,469 [RapidOCR] download_file.py:60: File exists and is valid: /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-29 14:04:00,469 [RapidOCR] main.py:50: Using /Users/utsav.talwa

[1/43] 10172013104127AMAIS-022andAmds.pdf


[INFO] 2026-07-29 14:04:00,521 [RapidOCR] download_file.py:60: File exists and is valid: /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth
[INFO] 2026-07-29 14:04:00,521 [RapidOCR] main.py:50: Using /Users/utsav.talwar/Desktop/UST/ust-demo/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1212.12it/s]


[2/43] 1122018124450PMAIS_55_and_Amd_1.pdf
[3/43] 1781154018_AIS-030 (Rev.2).pdf
[4/43] 1781154159_AIS-062 (Rev.2).pdf
[5/43] 1781860831_5_AIS_049_Rev.1_with_Amd_1 and 2.pdf
[6/43] 1784010622_AIS_030_ Rev1_with Amd 1 and 2.pdf
[7/43] 3272019101728AM7_AIS_063_and_Amds.pdf
[8/43] 330201534141PMAIS-011.pdf
[9/43] 413201592619AM13_7_AIS_049_Amd_2.pdf
[10/43] 5-15-2008-11-22-30-AM06_AIS-005.pdf
[11/43] 5192015105902AMAIS_002.PDF
[12/43] 5~15~2008~11~39~25~AM16 AIS-015.PDF
[13/43] 5~15~2008~12~01~48~PM31 AIS-030.pdf
[14/43] 5~15~2008~12~14~01~PM40 AIS-041.PDF
[15/43] 5~15~2008~12~18~36~PM43 AIS-044(Part 1).pdf
[16/43] 5~15~2008~12~20~09~PM44 AIS-044(Part 2).pdf
[17/43] 5~15~2008~12~21~32~PM45 AIS-044(Part 3).pdf
[18/43] 5~15~2008~1~05~57~PM58 AIS-062.pdf
[19/43] 7212015105501AM14_7_AIS_050_Amds.pdf
[20/43] 7302018102243AMAIS_018.pdf
[21/43] 77201442703PMAIS_051_Amd_1_and_2.pdf
[22/43] 7_AIS 001_Part_1_Rev_2_&_Corri_1F_3eeec486-eb12-4468-ba72-e42ef749a1c0.pdf
[23/43] 7_AIS_002_Part1_Rev2_&_Am

RapidOCR returned empty result!


[25/43] 918201455650PMAIS-034(Part1)(Rev1)andAmd1.pdf
[26/43] AIS-025 (Version 3) with Amd 1 to 8_2c627b8a-1b22-47a7-9e79-d162f42f2050.pdf
[27/43] AIS-026 (Version3) with Amd 1_303e6731-a999-47a5-99cc-9067c30605f4.pdf
[28/43] AIS-027 (Version 3)_with_Amd 1_7f0257bd-8335-4767-ad26-02c512aa8a9c.pdf
[29/43] AIS-034 (Rev.3) (Part 2)_Final_2e59fcf6-dd70-4a72-b06d-d2a485ee8640.pdf
[30/43] AIS-034 (Rev.3)(Part 1)_Final_473e3dc9-67fa-4420-ade6-1203d4567074.pdf
[31/43] AIS-052_08236d19-d9a3-4ef8-a50e-08f917ea40dc.pdf


RapidOCR returned empty result!
[WARNING] 2026-07-29 14:34:00,761 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-07-29 14:34:02,722 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!


[32/43] AIS-057_Rev_1_with Amd_57fb9704-c742-49c8-8cbc-5637bb2d783d.pdf
[33/43] AIS_041_Rev.1_with Amd 1_2c56319c-6228-4a3e-a29b-aa161b502807.pdf
[34/43] AIS_062_Rev_1_540d4496-bc33-4159-a432-5ce6892e2a92.pdf
[35/43] PUB_10~17~2011~12~05~20~PM~AIS-002_Part1_Rev1_F.pdf
[36/43] PUB_12~29~2010~12~18~27~PM~AIS-034(Part2)(Rev1)F.pdf
[37/43] PUB_2~27~2012~12~08~23~PM~AIS-014_Amds_1_and_2_and_3.pdf
[38/43] PUB_5~23~2011~4~48~45~PM~AIS-001withAmd1and2(60910).pdf
[39/43] PUB_5~23~2011~5~22~51~PM~AIS-034andAmd1and2.pdf
[40/43] PUB_5~23~2011~5~34~49~PM~AIS-057F.pdf
[41/43] PUB_5~6~2011~5~34~15~PM~AIS-001(Part1)(Rev.1)F.pdf
[42/43] PUB_5~6~2011~5~39~24~PM~AIS-001(Part2)(Rev.1)F.pdf
[43/43] PUB_5~6~2011~5~43~48~PM~AIS-002(Part2)(Rev.1)F.pdf
Wrote 3139 documents from 43 PDFs to ais_extracted/. Status: {'ok': 42, 'no_clauses': 1, 'error': 0}


[{'AIS_id': 'AIS-022',
  'source_file': '10172013104127AMAIS-022andAmds.pdf',
  'status': 'ok',
  'document_count': 9,
  'used_ocr': True,
  'error': None},
 {'AIS_id': 'AIS-055',
  'source_file': '1122018124450PMAIS_55_and_Amd_1.pdf',
  'status': 'ok',
  'document_count': 27,
  'used_ocr': True,
  'error': None},
 {'AIS_id': 'AIS-030',
  'source_file': '1781154018_AIS-030 (Rev.2).pdf',
  'status': 'ok',
  'document_count': 231,
  'used_ocr': True,
  'error': None},
 {'AIS_id': 'AIS-062',
  'source_file': '1781154159_AIS-062 (Rev.2).pdf',
  'status': 'ok',
  'document_count': 14,
  'used_ocr': True,
  'error': None},
 {'AIS_id': 'AIS-049',
  'source_file': '1781860831_5_AIS_049_Rev.1_with_Amd_1 and 2.pdf',
  'status': 'ok',
  'document_count': 25,
  'used_ocr': True,
  'error': None},
 {'AIS_id': 'AIS-030',
  'source_file': '1784010622_AIS_030_ Rev1_with Amd 1 and 2.pdf',
  'status': 'ok',
  'document_count': 135,
  'used_ocr': True,
  'error': None},
 {'AIS_id': 'AIS-063',
  'source_f

In [67]:
if WRITE_TO_MONGODB:

    if not MONGODB_URI:

        raise RuntimeError("Set MONGODB_URI before enabling MongoDB writes.")



    client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=10_000)

    try:

        client.admin.command("ping")

        collection = client[MONGODB_DATABASE][MONGODB_COLLECTION]

        collection.create_index(

            [("source_file", ASCENDING), ("rule", ASCENDING)],

            unique=True,

            name="source_file_rule_unique",

        )

        result = collection.bulk_write(

            [

                UpdateOne(

                    {

                        "source_file": document["source_file"],

                        "rule": document["rule"],

                    },

                    {"$set": document},

                    upsert=True,

                )

                for document in all_documents

            ],

            ordered=False,

        )

        print(

            f"MongoDB upsert complete: {result.upserted_count} inserted, "

            f"{result.modified_count} updated in "

            f"{MONGODB_DATABASE}.{MONGODB_COLLECTION}."

        )

    finally:

        client.close()

else:

    print("MongoDB write skipped. Review ais_extracted/, then enable WRITE_TO_MONGODB.")

MongoDB upsert complete: 3139 inserted, 0 updated in automotive_regulations.ais_rules.
